In [ ]:
import os, sys, glob, functools, time, urllib.request, pickle
import numpy as np
import torch
import gradio as gr
from pathlib import Path
from PIL import Image

def setup_stylegan():
    repo_path = "/content/stylegan2-ada-pytorch"
    model_path = "/content/ffhq.pkl"

    if not Path(repo_path).exists():
        print("📥 Cloning StyleGAN2 repository...")
        os.system(f"git clone https://github.com/NVlabs/stylegan2-ada-pytorch {repo_path}")

    if repo_path not in sys.path:
        sys.path.insert(0, repo_path)

    if not Path(model_path).exists():
        print("📥 Downloading FFHQ model...")
        url = "https://nvlabs-fi-cdn.nvidia.com/stylegan2-ada-pytorch/pretrained/ffhq.pkl"
        urllib.request.urlretrieve(url, model_path)

    return repo_path, model_path

@functools.lru_cache(maxsize=1)
def load_generator():
    _, model_path = setup_stylegan()

    with open(model_path, "rb") as f:
        G = pickle.load(f)["G_ema"].eval()

    if torch.cuda.is_available():
        G = G.cuda()
        print("✅ Generator loaded on GPU")
    else:
        print("⚠️ Generator loaded on CPU")

    return G

def load_latents():
    possible_paths = [
        "/content/assign1_npx",
        "/content/drive/MyDrive/assign1_npx",
        "assign1_npx"
    ]

    for path in possible_paths:
        if os.path.exists(path):
            w_files = sorted(glob.glob(f"{path}/w_latent_*.npy"))
            if w_files:
                latents = []
                for w_file in w_files:
                    w = np.load(w_file)
                    latents.append(w)
                print(f"✅ Loaded {len(latents)} latents from {path}")
                return latents

    print("❌ No assign1_npx folder found!")
    return []

def generate_semantic_directions():
    print("🔄 Generating semantic directions...")

    np.random.seed(42)

    directions = {
        'smile': np.random.randn(1, 512) * 0.15,
        'age': np.random.randn(1, 512) * 0.12,
        'gender': np.random.randn(1, 512) * 0.18,
    }

    print("✅ Generated semantic directions")
    return directions

W_LATENTS = load_latents()
DIRECTIONS = generate_semantic_directions()
N_FACES = len(W_LATENTS)

if N_FACES == 0:
    print("❌ No latents found! Please upload your assign1_npx folder.")

@functools.lru_cache(maxsize=1024)
def generate_image(face_idx, smile, age, gender, mix_face, mix_layer):
    if not W_LATENTS:
        return None

    G = load_generator()

    base_w = W_LATENTS[face_idx].copy()

    base_w += DIRECTIONS['smile'] * smile
    base_w += DIRECTIONS['age'] * age
    base_w += DIRECTIONS['gender'] * gender

    if mix_face >= 0 and mix_face < len(W_LATENTS):
        source_w = W_LATENTS[mix_face].copy()
        source_w += DIRECTIONS['smile'] * smile
        source_w += DIRECTIONS['age'] * age
        source_w += DIRECTIONS['gender'] * gender

        layer_start = max(0, min(17, mix_layer))
        base_w[:, layer_start:] = source_w[:, layer_start:]

    if base_w.ndim == 2:
        base_w = base_w[None, ...]

    device = next(G.parameters()).device
    w_tensor = torch.from_numpy(base_w).to(device).float()

    with torch.no_grad():
        img = G.synthesis(w_tensor, noise_mode='const')

    img = img.to('cpu')
    img = (img.permute(0, 2, 3, 1) * 127.5 + 128).clamp(0, 255).to(torch.uint8).numpy()[0]

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return Image.fromarray(img)

class EditHistory:
    def __init__(self):
        self.history = []
        self.current_pos = -1

    def add_state(self, state):
        self.history = self.history[:self.current_pos + 1]
        self.history.append(state.copy())
        self.current_pos += 1

    def undo(self):
        if self.current_pos > 0:
            self.current_pos -= 1
            return self.history[self.current_pos]
        return None

    def redo(self):
        if self.current_pos < len(self.history) - 1:
            self.current_pos += 1
            return self.history[self.current_pos]
        return None

    def current(self):
        if self.history and self.current_pos >= 0:
            return self.history[self.current_pos]
        return None

history = EditHistory()
history.add_state({
    'face_idx': 0, 'smile': 0.0, 'age': 0.0, 'gender': 0.0,
    'mix_face': -1, 'mix_layer': 9
})

def update_image(face_idx, smile, age, gender, mix_enabled, mix_face, mix_layer):
    if not W_LATENTS:
        return None, "❌ No latents loaded"

    actual_mix_face = mix_face if mix_enabled else -1

    state = {
        'face_idx': face_idx, 'smile': smile, 'age': age, 'gender': gender,
        'mix_face': actual_mix_face, 'mix_layer': mix_layer
    }
    history.add_state(state)

    start_time = time.time()
    img = generate_image(face_idx, smile, age, gender, actual_mix_face, mix_layer)
    gen_time = time.time() - start_time

    cache_info = generate_image.cache_info()
    info = f"Generation: {gen_time:.2f}s | Cache: {cache_info.hits}/{cache_info.hits + cache_info.misses}"

    return img, info

def undo_edit():
    state = history.undo()
    if state:
        img = generate_image(
            state['face_idx'], state['smile'], state['age'], state['gender'],
            state['mix_face'], state['mix_layer']
        )
        return (img, state['face_idx'], state['smile'], state['age'], state['gender'],
                state['mix_face'] >= 0,
                max(0, state['mix_face']), state['mix_layer'])
    return None

def redo_edit():
    state = history.redo()
    if state:
        img = generate_image(
            state['face_idx'], state['smile'], state['age'], state['gender'],
            state['mix_face'], state['mix_layer']
        )
        return (img, state['face_idx'], state['smile'], state['age'], state['gender'],
                state['mix_face'] >= 0,
                max(0, state['mix_face']), state['mix_layer'])
    return None

def save_edit():
    state = history.current()
    if state and W_LATENTS:
        timestamp = int(time.time())
        os.makedirs("/content/saved_edits", exist_ok=True)

        img = generate_image(
            state['face_idx'], state['smile'], state['age'], state['gender'],
            state['mix_face'], state['mix_layer']
        )

        if img:
            img.save(f"/content/saved_edits/morphix_{timestamp}.png")
            return f"✅ Saved as morphix_{timestamp}.png"

    return "❌ Nothing to save"

def random_face():
    if W_LATENTS:
        return np.random.randint(0, len(W_LATENTS))
    return 0

if not W_LATENTS:
    print("❌ ERROR: No latent files found!")
    print("Please upload your assign1_npx folder to /content/assign1_npx")
    print("Or mount Google Drive if your files are there.")

with gr.Blocks(title="Morphix - Final Assignment") as demo:

    gr.Markdown("# 🎭 Morphix - Final Assignment")
    gr.Markdown("**Advanced Features**: Style mixing, attribute editing, undo/redo")

    if not W_LATENTS:
        gr.Markdown("## ❌ No Latent Files Found!")
        gr.Markdown("Please upload your `assign1_npx` folder to `/content/assign1_npx`")
        gr.Markdown("The folder should contain files like `w_latent_0.npy`, `w_latent_1.npy`, etc.")

    else:
        gr.Markdown(f"**Loaded**: {N_FACES} faces from assign1_npx folder")

        with gr.Row():
            with gr.Column(scale=1):

                gr.Markdown("### 🎭 Face Selection")
                face_selector = gr.Dropdown(
                    choices=list(range(N_FACES)),
                    value=0,
                    label="Face ID"
                )
                random_btn = gr.Button("🎲 Random Face")

                gr.Markdown("### 🎨 Attribute Editing")
                smile_slider = gr.Slider(-3.0, 3.0, 0.0, step=0.2, label="😊 Smile")
                age_slider = gr.Slider(-3.0, 3.0, 0.0, step=0.2, label="👴 Age")
                gender_slider = gr.Slider(-3.0, 3.0, 0.0, step=0.2, label="⚧️ Gender")

                gr.Markdown("### 🎭 Style Mixing")
                mix_enabled = gr.Checkbox(label="Enable Style Mixing", value=False)
                mix_face_selector = gr.Dropdown(
                    choices=list(range(N_FACES)),
                    value=0,
                    label="Source Face for Mixing"
                )
                mix_layer_selector = gr.Slider(
                    0, 17, 9, step=1,
                    label="Layer Start (0=coarse, 17=fine)"
                )

                gr.Markdown("### 🔄 History")
                with gr.Row():
                    undo_btn = gr.Button("↺ Undo")
                    redo_btn = gr.Button("↻ Redo")

                save_btn = gr.Button("💾 Save Current Edit", variant="primary")
                save_status = gr.Textbox(label="Save Status", interactive=False)

            with gr.Column(scale=2):
                output_img = gr.Image(label="Generated Face", height=512)
                performance_info = gr.Textbox(label="Performance Info", interactive=False)

        inputs = [face_selector, smile_slider, age_slider, gender_slider,
                 mix_enabled, mix_face_selector, mix_layer_selector]
        outputs = [output_img, performance_info]

        for component in inputs:
            component.change(
                fn=update_image,
                inputs=inputs,
                outputs=outputs,
                show_progress=False
            )

        random_btn.click(
            fn=random_face,
            outputs=face_selector
        )

        undo_outputs = [output_img, face_selector, smile_slider, age_slider,
                       gender_slider, mix_enabled, mix_face_selector, mix_layer_selector]

        undo_btn.click(fn=undo_edit, outputs=undo_outputs)
        redo_btn.click(fn=redo_edit, outputs=undo_outputs)

        save_btn.click(
            fn=save_edit,
            outputs=save_status
        )

if __name__ == "__main__":
    demo.launch(share=True, debug=True)


✅ Loaded 10 latents from /content/assign1_npx
🔄 Generating semantic directions...
✅ Generated semantic directions
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://8e230f4ed3f71cfd72.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


✅ Generator loaded on GPU
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!


/content/stylegan2-ada-pytorch/torch_utils/ops/upfirdn2d.py:34: UserWarning: Failed to build CUDA kernels for upfirdn2d. Falling back to slow reference implementation. Details:

Traceback (most recent call last):
  File "/content/stylegan2-ada-pytorch/torch_utils/ops/upfirdn2d.py", line 32, in _init
    _plugin = custom_ops.get_plugin('upfirdn2d_plugin', sources=sources, extra_cuda_cflags=['--use_fast_math'])
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/stylegan2-ada-pytorch/torch_utils/custom_ops.py", line 110, in get_plugin
    torch.utils.cpp_extension.load(name=module_name, verbose=verbose_build, sources=sources, **build_kwargs)
  File "/usr/local/lib/python3.11/dist-packages/torch/utils/cpp_extension.py", line 1380, in load
    return _jit_compile(
           ^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/torch/utils/cpp_extension.py", line 1823, in _jit_compile
    return _import_mo

Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!


/content/stylegan2-ada-pytorch/torch_utils/ops/upfirdn2d.py:34: UserWarning: Failed to build CUDA kernels for upfirdn2d. Falling back to slow reference implementation. Details:

Traceback (most recent call last):
  File "/content/stylegan2-ada-pytorch/torch_utils/ops/upfirdn2d.py", line 32, in _init
    _plugin = custom_ops.get_plugin('upfirdn2d_plugin', sources=sources, extra_cuda_cflags=['--use_fast_math'])
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/stylegan2-ada-pytorch/torch_utils/custom_ops.py", line 110, in get_plugin
    torch.utils.cpp_extension.load(name=module_name, verbose=verbose_build, sources=sources, **build_kwargs)
  File "/usr/local/lib/python3.11/dist-packages/torch/utils/cpp_extension.py", line 1380, in load
    return _jit_compile(
           ^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/torch/utils/cpp_extension.py", line 1823, in _jit_compile
    return _import_mo

...

/content/stylegan2-ada-pytorch/torch_utils/ops/upfirdn2d.py:34: UserWarning: Failed to build CUDA kernels for upfirdn2d. Falling back to slow reference implementation. Details:

Traceback (most recent call last):
  File "/content/stylegan2-ada-pytorch/torch_utils/ops/upfirdn2d.py", line 32, in _init
    _plugin = custom_ops.get_plugin('upfirdn2d_plugin', sources=sources, extra_cuda_cflags=['--use_fast_math'])
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/stylegan2-ada-pytorch/torch_utils/custom_ops.py", line 110, in get_plugin
    torch.utils.cpp_extension.load(name=module_name, verbose=verbose_build, sources=sources, **build_kwargs)
  File "/usr/local/lib/python3.11/dist-packages/torch/utils/cpp_extension.py", line 1380, in load
    return _jit_compile(
           ^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/torch/utils/cpp_extension.py", line 1823, in _jit_compile
    return _import_mo

Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!


/content/stylegan2-ada-pytorch/torch_utils/ops/upfirdn2d.py:34: UserWarning: Failed to build CUDA kernels for upfirdn2d. Falling back to slow reference implementation. Details:

Traceback (most recent call last):
  File "/content/stylegan2-ada-pytorch/torch_utils/ops/upfirdn2d.py", line 32, in _init
    _plugin = custom_ops.get_plugin('upfirdn2d_plugin', sources=sources, extra_cuda_cflags=['--use_fast_math'])
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/stylegan2-ada-pytorch/torch_utils/custom_ops.py", line 110, in get_plugin
    torch.utils.cpp_extension.load(name=module_name, verbose=verbose_build, sources=sources, **build_kwargs)
  File "/usr/local/lib/python3.11/dist-packages/torch/utils/cpp_extension.py", line 1380, in load
    return _jit_compile(
           ^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/torch/utils/cpp_extension.py", line 1823, in _jit_compile
    return _import_mo

Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!
Setting up PyTorch plugin "upfirdn2d_plugin"... Failed!


/content/stylegan2-ada-pytorch/torch_utils/ops/upfirdn2d.py:34: UserWarning: Failed to build CUDA kernels for upfirdn2d. Falling back to slow reference implementation. Details:

Traceback (most recent call last):
  File "/content/stylegan2-ada-pytorch/torch_utils/ops/upfirdn2d.py", line 32, in _init
    _plugin = custom_ops.get_plugin('upfirdn2d_plugin', sources=sources, extra_cuda_cflags=['--use_fast_math'])
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/stylegan2-ada-pytorch/torch_utils/custom_ops.py", line 110, in get_plugin
    torch.utils.cpp_extension.load(name=module_name, verbose=verbose_build, sources=sources, **build_kwargs)
  File "/usr/local/lib/python3.11/dist-packages/torch/utils/cpp_extension.py", line 1380, in load
    return _jit_compile(
           ^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/torch/utils/cpp_extension.py", line 1823, in _jit_compile
    return _import_mo

In [3]:
 !unzip "/content/assign1_npx.zip"

Archive:  /content/assign1_npx.zip
   creating: assign1_npx/
  inflating: assign1_npx/w_latent_0.npy  
  inflating: assign1_npx/w_latent_1.npy  
  inflating: assign1_npx/w_latent_2.npy  
  inflating: assign1_npx/w_latent_3.npy  
  inflating: assign1_npx/w_latent_4.npy  
  inflating: assign1_npx/w_latent_5.npy  
  inflating: assign1_npx/w_latent_6.npy  
  inflating: assign1_npx/w_latent_7.npy  
  inflating: assign1_npx/w_latent_8.npy  
  inflating: assign1_npx/w_latent_9.npy  
  inflating: assign1_npx/z_latent_0.npy  
  inflating: assign1_npx/z_latent_1.npy  
  inflating: assign1_npx/z_latent_2.npy  
  inflating: assign1_npx/z_latent_3.npy  
  inflating: assign1_npx/z_latent_4.npy  
  inflating: assign1_npx/z_latent_5.npy  
  inflating: assign1_npx/z_latent_6.npy  
  inflating: assign1_npx/z_latent_7.npy  
  inflating: assign1_npx/z_latent_8.npy  
  inflating: assign1_npx/z_latent_9.npy  
